In [1]:
import pandas as pd
import os
import re

In [2]:
postings_path = "/Users/karmesh/Desktop/ResumeBoost-and-JobFit/datasets/postings.csv"

# Load the Job Postings CSV file into a Pandas DataFrame
postings_df = pd.read_csv(postings_path)

# Display the first few rows of the dataset to understand its structure
postings_df.head() 

FileNotFoundError: [Errno 2] No such file or directory: '/Users/karmesh/Desktop/ResumeBoost-and-JobFit/datasets/postings.csv'

In [ ]:
postings_df.shape

In [ ]:
postings_df.head()

In [ ]:
postings_sample_df = postings_df.sample(1000)
postings_sample_df.shape

In [ ]:
postings_sample_df.head()

In [ ]:
def preprocess_text(text):
    """Preprocess text by converting to lowercase, removing special characters, and handling NaN."""
    if pd.isnull(text):
        return ""
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove special characters
    return text.strip()

def combine_features(row):
    """Combine relevant features into a single string."""
    features = []
    for col in ['title', 'description', 'skills_desc']:
        if not pd.isnull(row[col]):
            features.append(f"{col.capitalize()}: {preprocess_text(row[col])}\n")
    return ' '.join(features)

# Apply preprocessing and feature combination
postings_sample_df['combined_features'] = postings_sample_df.apply(combine_features, axis=1)

# Display a sample of the combined data
postings_sample_df[['job_id', 'combined_features']].head()

In [ ]:
print(postings_sample_df.iloc[0]['combined_features'])

In [ ]:
import pandas as pd
resume_path = '/Users/karmesh/Desktop/ResumeBoost-and-JobFit/datasets/Resume.csv'

resume_df = pd.read_csv(resume_path)

# Display the first few rows of the dataset to understand its structure
resume_df.head()

In [ ]:
def preprocess_resume_text(row):
    """Preprocess resume text by converting to lowercase, removing special characters, and handling NaN."""
    text = row.get('Resume_str', '') 
    if pd.isnull(text):
        return ""
    text = re.sub(r'[^\w\s,+./-]', '', text)  # Remove unwanted characters but retain important ones like +, , . / -
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespaces
    text = text.strip()  # Trim leading/trailing whitespace
    text = text.lower() # Normalize to lowercase
    
    return text

# Apply preprocessing
resume_df['preprocessed_resume'] = resume_df.apply(preprocess_resume_text, axis=1)

# Display the first 5 rows
resume_df[['ID', 'preprocessed_resume']].head()

In [ ]:
print(resume_df.iloc[0]['preprocessed_resume'])

In [ ]:
%pip install -U voyageai

In [148]:
import config
import voyageai

In [ ]:
import voyageai
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from voyage_config import VOYAGE_API_KEY

# Initialize the Voyage API client
vo = voyageai.Client(api_key=VOYAGE_API_KEY)
embedding_model = "voyage-3-lite"

# Function to generate embeddings in batches
def generate_embeddings(texts, batch_size=128):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        try:
            result = vo.embed(batch, model=embedding_model, input_type="document")
            all_embeddings.extend(result.embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            return None
    return all_embeddings

# Function to find similar jobs
def find_similar_jobs(resume_embedding, postings_df, top_n=5):
    """Finds the most similar job postings to a given resume embedding."""
    job_embeddings = np.array(postings_df['embeddings'].tolist())
    similarities = cosine_similarity([resume_embedding], job_embeddings)[0]
    top_indices = similarities.argsort()[-top_n:][::-1]
    return top_indices

# Main execution
if __name__ == "__main__":
    # Load YOUR resume dataset (replace with your actual path)
    resume_path = "/Users/karmesh/Desktop/ResumeBoost-and-JobFit/datasets/Resume.csv"  # Update this path
    resume_df = pd.read_csv(resume_path)
    
    # Load YOUR job postings dataset (already loaded as postings_df in previous steps)
    postings_path = "/Users/karmesh/Desktop/ResumeBoost-and-JobFit/datasets/postings.csv"
    postings_df = pd.read_csv(postings_path)

    # Preprocess job postings (combine title and description)
    postings_df['combined_features'] = postings_df['title'] + " " + postings_df['description']
    postings_df['combined_features'] = postings_df['combined_features'].fillna('')

    # Generate embeddings for job postings
    postings_df['embeddings'] = generate_embeddings(postings_df['combined_features'].tolist())

    # Generate embedding for a specific resume (e.g., the first resume in your dataset)
    resume_text = resume_df.iloc[0]['preprocessed_resume']
    resume_embedding = generate_embeddings([resume_text])[0]

    # Find top 5 similar jobs
    similar_job_indices = find_similar_jobs(resume_embedding, postings_df)

    # Print results
    print("Top job matches for resume:")
    print("-" * 50)
    print(f"Resume ID: {resume_df.iloc[0]['ID']}")
    print(f"Resume Content: {resume_df.iloc[0]['preprocessed_resume']}")
    print("-" * 50)

    for i, job_index in enumerate(similar_job_indices):
        job_posting = postings_df.iloc[job_index]
        print(f"Match {i+1}:")
        print(f"Job ID: {job_posting['job_id']}")
        print(f"Title: {job_posting['title']}")
        print(f"Similarity Score: {cosine_similarity([resume_embedding], [job_posting['embeddings']])[0][0]:.4f}")
        print(f"Description: {job_posting['description'][:200]}...")  # Truncate for readability
        print("-" * 50)

In [ ]:
resume_df.head()

In [ ]:
%pip install sentence-transformers scikit-learn


In [ ]:

import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ----------------------
# 1. Load Model
# ----------------------
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight model

# ----------------------
# 2. Generate Embeddings Locally
# ----------------------
def generate_embeddings(texts):
    return model.encode(texts, convert_to_tensor=False)

# ----------------------
# 3. Similarity Search (Modified)
# ----------------------
def find_job_matches(resume_id, top_n=5):
    # Load data (same as before)
    postings_df, resume_df = load_and_preprocess_data()
    
    # Generate embeddings
    job_embeddings = generate_embeddings(postings_df['combined_features'].tolist())
    resume_embedding = generate_embeddings([resume_df.loc[resume_id, 'processed']])[0]
    
    # Calculate similarities
    similarities = cosine_similarity([resume_embedding], job_embeddings)[0]
    top_indices = similarities.argsort()[-top_n:][::-1]
    
    # Return results (same as before)
    return format_results(top_indices, postings_df, similarities)